# 第一阶段 步骤04：数值微分

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第四步。

---

## 核心目标

在实现反向传播之前，先用**数值微分（numerical differentiation）**近似求导——即用极小的差分来近似导数。

它为"求导"建立直觉，也是理解后续反向传播为何更优的对照。

## 4.1 什么是导数

导数表示函数在某一点的**变化率**（斜率）：

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

问题在于：**计算机无法处理极限**（h 无限趋近 0 做不到）。所以只能取一个很小的 h（如 `eps = 1e-4`）来近似。

## 4.2 数值微分的实现

用微小差值近似导数，就是**数值微分**。有两种差分方式：

| 方式 | 公式 | 说明 |
| --- | --- | --- |
| 前向差分 | $\frac{f(x+h) - f(x)}{h}$ | 用 x 与 x+h 两点斜率，误差较大 |
| 中心差分 | $\frac{f(x+h) - f(x-h)}{2h}$ | 用 x-h 与 x+h 两点斜率，误差更小 |

中心差分比前向差分更接近真实导数，可用**泰勒展开**证明。因此本书采用中心差分。

In [ ]:
import numpy as np

# 承接步骤03：Variable、Function、Square、Exp
class Variable:
    def __init__(self, data):
        self.data = data

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        return output

    def forward(self, x):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2

class Exp(Function):
    def forward(self, x):
        return np.exp(x)

# 4.2 数值微分：中心差分近似
def numerical_diff(f, x, eps=1e-4):
    x0 = Variable(x.data - eps)
    x1 = Variable(x.data + eps)
    y0 = f(x0)
    y1 = f(x1)
    return (y1.data - y0.data) / (2 * eps)

In [ ]:
# 验证：y = x^2 的导数应为 2x
f = Square()

x = Variable(np.array(2.0))
dy = numerical_diff(f, x)
print(dy)   # 4.000000000004（精确值 4.0）

x0 = Variable(np.array(0.0))
print(numerical_diff(f, x0))   # 0.0

## 4.3 复合函数的导数

数值微分同样适用于**复合函数**。把 `C(B(A(x)))` 当作一个函数 `f`，传入 `numerical_diff` 即可自动求导。

到这一步，其实已经实现了"**自动**求导"：只要用代码定义好计算，程序就能求出导数——无论多复杂的函数（只要可微）。

In [ ]:
# 对复合函数 y = (e^(x^2))^2 求导
A = Square()
B = Exp()
C = Square()

def f(x):
    return C(B(A(x)))   # 把复合函数当成一个整体

x = Variable(np.array(0.5))
dy = numerical_diff(f, x)
print(dy)   # 3.2974426293330694

## 4.4 数值微分存在的问题

| 问题 | 说明 |
| --- | --- |
| 结果有误差 | 多数情况误差很小，但个别情况下会很大（精度丢失） |
| 计算效率低 | 每个参数都要做两次前向计算 |

## 这一步的"为什么"

数值微分不是终点，而是**过渡**：它让我们先理解"求导"这件事，同时暴露效率问题——这正是下一步引入**反向传播**的动机。

反向传播能用一次前向 + 一次反向，高效地同时求出所有变量的导数。

---

> 预告：步骤5 正式进入**反向传播**，给 `Function` 加上 `backward` 方法。